In [ ]:
!pip install shap openpyxl xgboost


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import shap


csv_files = {
    "PIMA_Dataset": "diabetes.csv",
    "Medical_City_Dataset": "Dataset of Diabetes .csv"
}

print("Both CSV files loading framework active...")


Both CSV files loading framework active...


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import shap

print("="*60)
print("COMPUTING EMPIRICAL METRICS FOR: PIMA_Dataset")
print("="*60)

try:
    raw_dataframe = pd.read_csv("diabetes.csv")
    print(f"Dataframe connection validated. Dimensions: {raw_dataframe.shape}")

    target_variable = None
    for current_col in raw_dataframe.columns:
        if current_col.lower().strip() in ['outcome', 'class', 'target', 'result']:
            target_variable = current_col
            break
    if target_variable is None:
        target_variable = raw_dataframe.columns[-1]

    print(f"Target vector channel established: '{target_variable}'")

    matrix_x = raw_dataframe.drop(columns=[target_variable])
    vector_y = raw_dataframe[target_variable]

    if hasattr(vector_y, 'dtype') and (vector_y.dtype == 'object' or str(vector_y.dtype).startswith('cat')):
        vector_y, _ = pd.factorize(vector_y)

    matrix_x = matrix_x.select_dtypes(include=[np.number])
    matrix_x = matrix_x.fillna(matrix_x.median())

    x_train, x_test, y_train, y_test = train_test_split(matrix_x, vector_y, test_size=0.2, random_state=42)

    feature_scaler = StandardScaler()
    x_train_scaled = feature_scaler.fit_transform(x_train)
    x_test_scaled = feature_scaler.transform(x_test)

    processed_train_df = pd.DataFrame(x_train_scaled, columns=matrix_x.columns)
    processed_test_df = pd.DataFrame(x_test_scaled, columns=matrix_x.columns)

    baseline_lr_estimator = LogisticRegression(max_iter=1000, random_state=42)
    baseline_lr_estimator.fit(processed_train_df, y_train)
    lr_predictions = baseline_lr_estimator.predict(processed_test_df)

    ensemble_xgb_estimator = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    ensemble_xgb_estimator.fit(processed_train_df, y_train)
    xgb_predictions = ensemble_xgb_estimator.predict(processed_test_df)

    print("\nFinal Quantitative Summary:")
    print(f"-> Logistic Regression Accuracy : {accuracy_score(y_test, lr_predictions) * 100:.2f}%")
    print(f"-> Advanced XGBoost Accuracy     : {accuracy_score(y_test, xgb_predictions) * 100:.2f}%")

    print("\nComputing localized attribution metrics via SHAP kernels...")
    shap_tree_interpreter = shap.TreeExplainer(ensemble_xgb_estimator)
    extracted_shap_values = shap_tree_interpreter(processed_test_df)

    computed_mean_weights = np.abs(extracted_shap_values.values).mean(axis=0)
    clinical_importance_map = pd.DataFrame({'Feature': matrix_x.columns, 'Impact_Metric': computed_mean_weights})
    clinical_importance_map = clinical_importance_map.sort_values(by='Impact_Metric', ascending=False)

    print("\nPrimary 3 Clinical Indicators Manifested:")
    print(clinical_importance_map.head(3).to_string(index=False))

except Exception as operational_error:
    print(f"Structural execution error: {str(operational_error)}")



COMPUTING EMPIRICAL METRICS FOR: PIMA_Dataset
Dataframe connection validated. Dimensions: (768, 9)
Target vector channel established: 'Outcome'

Final Quantitative Summary:
-> Logistic Regression Accuracy : 75.32%
-> Advanced XGBoost Accuracy     : 72.08%

Computing localized attribution metrics via SHAP kernels...

Primary 3 Clinical Indicators Manifested:
Feature  Impact_Metric
Glucose       1.890491
    BMI       1.223269
    Age       1.113508


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import shap

print("="*60)
print("COMPUTING FIX FOR: Medical_City_Dataset")
print("="*60)

try:
    raw_dataframe = pd.read_csv("Dataset of Diabetes .csv")

    # Trim column names to remove accidental hidden spaces
    raw_dataframe.columns = raw_dataframe.columns.str.strip()

    target_variable = 'CLASS' if 'CLASS' in raw_dataframe.columns else raw_dataframe.columns[-1]
    print(f"Target vector channel established: '{target_variable}'")

    # Handle text columns safely using LabelEncoder
    for col in raw_dataframe.columns:
        if raw_dataframe[col].dtype == 'object':
            le = LabelEncoder()
            raw_dataframe[col] = le.fit_transform(raw_dataframe[col].astype(str))

    # Handle missing values
    raw_dataframe = raw_dataframe.fillna(raw_dataframe.median())

    matrix_x = raw_dataframe.drop(columns=[target_variable])
    vector_y = raw_dataframe[target_variable]

    x_train, x_test, y_train, y_test = train_test_split(matrix_x, vector_y, test_size=0.2, random_state=42)

    feature_scaler = StandardScaler()
    x_train_scaled = feature_scaler.fit_transform(x_train)
    x_test_scaled = feature_scaler.transform(x_test)

    processed_train_df = pd.DataFrame(x_train_scaled, columns=matrix_x.columns)
    processed_test_df = pd.DataFrame(x_test_scaled, columns=matrix_x.columns)

    baseline_lr_estimator = LogisticRegression(max_iter=1000, random_state=42)
    baseline_lr_estimator.fit(processed_train_df, y_train)
    lr_predictions = baseline_lr_estimator.predict(processed_test_df)

    ensemble_xgb_estimator = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    ensemble_xgb_estimator.fit(processed_train_df, y_train)
    xgb_predictions = ensemble_xgb_estimator.predict(processed_test_df)

    print("\nFinal Quantitative Summary:")
    print(f"-> Logistic Regression Accuracy : {accuracy_score(y_test, lr_predictions) * 100:.2f}%")
    print(f"-> Advanced XGBoost Accuracy     : {accuracy_score(y_test, xgb_predictions) * 100:.2f}%")

    print("\nComputing localized attribution metrics via SHAP kernels...")
    shap_tree_interpreter = shap.TreeExplainer(ensemble_xgb_estimator)

    # Flattening execution to remove the 1D dimension error
    extracted_shap_values = shap_tree_interpreter.shap_values(processed_test_df)

    # Handle multi-class or single-class arrays gracefully
    if isinstance(extracted_shap_values, list):
        computed_mean_weights = np.mean([np.abs(val).mean(axis=0) for val in extracted_shap_values], axis=0)
    else:
        if len(extracted_shap_values.shape) == 3:
            computed_mean_weights = np.abs(extracted_shap_values).mean(axis=(0, 2))
        else:
            computed_mean_weights = np.abs(extracted_shap_values).mean(axis=0)

    clinical_importance_map = pd.DataFrame({'Feature': matrix_x.columns, 'Impact_Metric': computed_mean_weights.flatten()})
    clinical_importance_map = clinical_importance_map.sort_values(by='Impact_Metric', ascending=False)

    print("\nPrimary 3 Clinical Indicators Manifested:")
    print(clinical_importance_map.head(3).to_string(index=False))

except Exception as operational_error:
    print(f"Structural execution error: {str(operational_error)}")



COMPUTING FIX FOR: Medical_City_Dataset
Target vector channel established: 'CLASS'

Final Quantitative Summary:
-> Logistic Regression Accuracy : 94.00%
-> Advanced XGBoost Accuracy     : 98.50%

Computing localized attribution metrics via SHAP kernels...

Primary 3 Clinical Indicators Manifested:
Feature  Impact_Metric
  HbA1c       1.126306
    BMI       0.690133
   Chol       0.340447


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Dataset paths setup
verified_paths = {
    "PIMA_Dataset": "diabetes.csv",
    "Medical_City_Dataset": "Dataset of Diabetes .csv"
}

print("="*60)
print("RUNNING MODEL: Decision Tree Classifier")
print("="*60)

for label, path in verified_paths.items():
    try:
        raw_dataframe = pd.read_csv(path)
        raw_dataframe.columns = raw_dataframe.columns.str.strip()

        # Identify target
        target_variable = 'CLASS' if 'CLASS' in raw_dataframe.columns else ('Outcome' if 'Outcome' in raw_dataframe.columns else raw_dataframe.columns[-1])

        matrix_x = raw_dataframe.drop(columns=[target_variable])
        vector_y = raw_dataframe[target_variable]

        # Encode text data
        for col in matrix_x.columns:
            if matrix_x[col].dtype == 'object':
                le = LabelEncoder()
                matrix_x[col] = le.fit_transform(matrix_x[col].astype(str))
        if hasattr(vector_y, 'dtype') and (vector_y.dtype == 'object' or str(vector_y.dtype).startswith('cat')):
            le_y = LabelEncoder()
            vector_y = le_y.fit_transform(vector_y.astype(str))

        matrix_x = matrix_x.select_dtypes(include=[np.number]).fillna(matrix_x.median())
        x_train, x_test, y_train, y_test = train_test_split(matrix_x, vector_y, test_size=0.2, random_state=42)

        # Scale features
        feature_scaler = StandardScaler()
        x_train_scaled = feature_scaler.fit_transform(x_train)
        x_test_scaled = feature_scaler.transform(x_test)

        # Train Decision Tree
        dt_model = DecisionTreeClassifier(random_state=42)
        dt_model.fit(x_train_scaled, y_train)
        dt_preds = dt_model.predict(x_test_scaled)

        print(f"{label} Accuracy -> Decision Tree: {accuracy_score(y_test, dt_preds) * 100:.2f}%")

    except Exception as error:
        print(f"Error for {label}: {str(error)}")



RUNNING MODEL: Decision Tree Classifier
PIMA_Dataset Accuracy -> Decision Tree: 74.68%
Medical_City_Dataset Accuracy -> Decision Tree: 96.50%


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Dataset paths setup
verified_paths = {
    "PIMA_Dataset": "diabetes.csv",
    "Medical_City_Dataset": "Dataset of Diabetes .csv"
}

print("="*60)
print("RUNNING MODEL: Random Forest Classifier")
print("="*60)

for label, path in verified_paths.items():
    try:
        raw_dataframe = pd.read_csv(path)
        raw_dataframe.columns = raw_dataframe.columns.str.strip()

        # Identify target
        target_variable = 'CLASS' if 'CLASS' in raw_dataframe.columns else ('Outcome' if 'Outcome' in raw_dataframe.columns else raw_dataframe.columns[-1])

        matrix_x = raw_dataframe.drop(columns=[target_variable])
        vector_y = raw_dataframe[target_variable]

        # Encode text data
        for col in matrix_x.columns:
            if matrix_x[col].dtype == 'object':
                le = LabelEncoder()
                matrix_x[col] = le.fit_transform(matrix_x[col].astype(str))
        if hasattr(vector_y, 'dtype') and (vector_y.dtype == 'object' or str(vector_y.dtype).startswith('cat')):
            le_y = LabelEncoder()
            vector_y = le_y.fit_transform(vector_y.astype(str))

        matrix_x = matrix_x.select_dtypes(include=[np.number]).fillna(matrix_x.median())
        x_train, x_test, y_train, y_test = train_test_split(matrix_x, vector_y, test_size=0.2, random_state=42)

        # Scale features
        feature_scaler = StandardScaler()
        x_train_scaled = feature_scaler.fit_transform(x_train)
        x_test_scaled = feature_scaler.transform(x_test)

        # Train Random Forest
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(x_train_scaled, y_train)
        rf_preds = rf_model.predict(x_test_scaled)

        print(f"{label} Accuracy -> Random Forest: {accuracy_score(y_test, rf_preds) * 100:.2f}%")

    except Exception as error:
        print(f"Error for {label}: {str(error)}")


RUNNING MODEL: Random Forest Classifier
PIMA_Dataset Accuracy -> Random Forest: 72.08%
Medical_City_Dataset Accuracy -> Random Forest: 99.00%
